# VRAM probe

Runs a few real training iterations and a few predictions for every model x input size x candidate batch size, and records peak GPU memory and speed. The recommendation table at the end gives, per run, the batch size and gradient accumulation to put into `configs/runs/<run_id>.yaml`, and an estimate of the training time against the 2-hour cap.

- Logic: `iiss.models.probe`. Results: `results/vram_probe/` (keep the outputs of this notebook: they are evidence).
- Run by the user on the local GPU (about 20-30 min for the full matrix). Close other GPU programs first.
- `SMOKE = True`: S=320 only, smallest batch, 5 iterations, 16 images; writes to `runs/smoke/`.

## Parameters

In [1]:
MODELS = ['yolo', 'maskrcnn', 'm2f']
SIZES = [320, 640, 960]
BATCHES = {'yolo': [4, 8, 16], 'maskrcnn': [1, 2, 4], 'm2f': [1, 2, 4]}
ITERS = 20            # training iterations per probe
WARMUP = 3            # first iterations not timed
INFER_IMAGES = 10     # valid images for the inference timing
BUDGET_MB = 5600      # max training peak (reserved) that counts as fitting on 6 GB
OUT_DIR = 'results/vram_probe'
FRESH = True          # delete the previous probe.csv first
SMOKE = False

In [2]:
import pandas as pd
import torch
from IPython.display import display

from iiss.models.common import apply_smoke
from iiss.models.probe import probe_matrix, recommend
from iiss.utils.config import load_run_config
from iiss.utils.log import setup_logging
from iiss.utils.paths import repo_root, resolve_path

setup_logging()
ROOT = repo_root()
if SMOKE:
    SIZES, ITERS, WARMUP, INFER_IMAGES = [320], 5, 1, 3
    BATCHES = {m: [min(b)] for m, b in BATCHES.items()}
    OUT_DIR = 'runs/smoke/vram_probe'
out_dir = resolve_path(OUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
if FRESH:
    (out_dir / 'probe.csv').unlink(missing_ok=True)
print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

2.14.1+cu126 NVIDIA GeForce RTX 4050 Laptop GPU


## Run configs to probe

In [3]:
configs = []
for model in MODELS:
    for size in SIZES:
        cfg = load_run_config(ROOT / 'configs' / 'runs' / f'{model}_s{size}_seed42.yaml')
        configs.append(apply_smoke(cfg) if SMOKE else cfg)
[c['run_id'] for c in configs]

['yolo_s320_seed42',
 'yolo_s640_seed42',
 'yolo_s960_seed42',
 'maskrcnn_s320_seed42',
 'maskrcnn_s640_seed42',
 'maskrcnn_s960_seed42',
 'm2f_s320_seed42',
 'm2f_s640_seed42',
 'm2f_s960_seed42']

## Probe

One row per model x S x batch. Larger batches are skipped once a smaller one runs out of memory.

In [4]:
results = probe_matrix(configs, BATCHES, out_dir / 'probe.csv',
                       iters=ITERS, warmup=WARMUP, infer_images=INFER_IMAGES)
display(results)

2026-11-23 00:25:41 INFO    iiss.models.probe: Probing yolo S=320 batch=4


New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.215  Python-3.12.4 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Syauqi Nabil\Lectures\University\Semester 5\Computer Vision\Task\Instance Segmentation\indoor-instance-seg-shift\dataset\derived\yolo\data.yaml, degrees=0.0, deterministic=False, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, i

c:\Users\Syauqi Nabil\Lectures\University\Semester 5\Computer Vision\Task\Instance Segmentation\indoor-instance-seg-shift\.venv\Lib\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
c:\Users\Syauqi Nabil\Lectures\University\Semester 5\Computer Vision\Task\Instance Segmentation\indoor-instance-seg-shift\.venv\Lib\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
c:\Users\Syauqi Nabil\Lectures\University\Semester 5\Computer Vision\Task\Instance Segmentation\indoor-instance-seg-shift\.venv\Lib\site-packages\torch\jit\_trace.py:1022: UserWarning: The input to trace is already a ScriptModule, tracing it is a no-op. Returning the object as is.
  traced_func = _trace_impl(


TensorBoard: model graph visualization added 
Using 397 train, 166 val images for fraction=1.0 at imgsz=320
Using 4 dataloader workers
Logging results to C:\Users\Syauqi Nabil\Lectures\University\Semester 5\Computer Vision\Task\Instance Segmentation\indoor-instance-seg-shift\runs\vram_probe\yolo_s320_seed42_b4
Starting training for 1 epochs...

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
        1/1     0.641G      1.765      4.542      4.663      1.645          0         67        320: 19% ━━────────── 19/100 5.0it/s 40.1s<16.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.6s/it 53.6s0.3s0
                   all        166       2021      0.354      0.161      0.145       0.11      0.349       0.14      0.144     0.0963

1 epochs completed in 0.032 hours.
Optimizer stripped from C:\Users\Syauqi Nabil\Lectures\Univers

2026-11-23 00:28:35 INFO    iiss.models.probe: -> {'model': 'yolo', 'input_size': 320, 'batch': 4, 'iters_timed': 16, 'oom': False, 'peak_train_mb': 656.0, 's_per_iter': 0.224, 'infer_ms': 98.6, 'peak_infer_mb': 130.0, 'note': ''} (174 s)
2026-11-23 00:28:35 INFO    iiss.models.probe: Probing yolo S=320 batch=8


New https://pypi.org/project/ultralytics/8.4.216 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.215  Python-3.12.4 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Syauqi Nabil\Lectures\University\Semester 5\Computer Vision\Task\Instance Segmentation\indoor-instance-seg-shift\dataset\derived\yolo\data.yaml, degrees=0.0, deterministic=False, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, i

... (41 outputs left out)

TensorBoard: model graph visualization added 
Using 397 train, 166 val images for fraction=1.0 at imgsz=960
Using 4 dataloader workers
Logging results to C:\Users\Syauqi Nabil\Lectures\University\Semester 5\Computer Vision\Task\Instance Segmentation\indoor-instance-seg-shift\runs\vram_probe\yolo_s960_seed42_b16
Starting training for 1 epochs...


c:\Users\Syauqi Nabil\Lectures\University\Semester 5\Computer Vision\Task\Instance Segmentation\indoor-instance-seg-shift\.venv\Lib\site-packages\torch\jit\_trace.py:1022: UserWarning: The input to trace is already a ScriptModule, tracing it is a no-op. Returning the object as is.
  traced_func = _trace_impl(


WARNING CUDA out of memory with batch=16. Reducing to batch=8 and retrying (1/3).
: 0% ──────────── 0/25  24.4s


AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.


## Recommendation

`batch` keeps the configured effective batch (with `accumulate`); `max_fit_batch` is the largest batch that fits at all. `est_train_h` is a planning estimate for the full epoch budget including validation; `cap_risk` = above 2 h.

In [ ]:
rec = recommend(results, {c['run_id']: c for c in configs}, budget_mb=BUDGET_MB)
rec.to_csv(out_dir / 'recommendation.csv', index=False)
display(rec)

## Next step

Send `results/vram_probe/recommendation.csv` (or this table) back to Claude Code. Batch overrides then go into `configs/runs/<run_id>.yaml`, and the epoch budgets are confirmed or adjusted per model (identical across S).